In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

# If the notebook is launched from inside notebooks/, move one level up.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)
print("Metadata:", METADATA_DIR)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Project root: /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape
Processed data: /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/data/processed
Metadata: /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/data/metadata
pandas: 3.0.2
numpy: 2.4.3


In [2]:
required_files = {
    "wo_counts": PROCESSED_DIR / "wo_counts.csv",
    "pooled_counts": PROCESSED_DIR / "WO_genotype_counts_per_library_condition.csv",
    "landscape": PROCESSED_DIR / "WO_landscape_per_library.csv",
    "landscape_pvalues": PROCESSED_DIR / "WO_landscape_with_pvalues.csv",
    "volcano_source": PROCESSED_DIR / "WO_landscape_ddelta_posneg_volcano_source.csv",
    "position_manifest": METADATA_DIR / "position_table_manifest.csv",
}

for name, path in required_files.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:20s} {status:8s} {path}")

wo_counts            FOUND    /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/data/processed/wo_counts.csv
pooled_counts        FOUND    /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/data/processed/WO_genotype_counts_per_library_condition.csv
landscape            FOUND    /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/data/processed/WO_landscape_per_library.csv
landscape_pvalues    FOUND    /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/data/processed/WO_landscape_with_pvalues.csv
volcano_source       FOUND    /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/data/processed/WO_landscape_ddelta_posneg_volcano_source.csv
position_manifest    FOUND    /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/data/metadata/position_table_manifest.csv


In [3]:
wo_counts = pd.read_csv(required_files["wo_counts"])
pooled_counts = pd.read_csv(required_files["pooled_counts"])
landscape = pd.read_csv(required_files["landscape"])
landscape_pvalues = pd.read_csv(required_files["landscape_pvalues"])
volcano_source = pd.read_csv(required_files["volcano_source"])
position_manifest = pd.read_csv(required_files["position_manifest"])

tables = {
    "wo_counts": wo_counts,
    "pooled_counts": pooled_counts,
    "landscape": landscape,
    "landscape_pvalues": landscape_pvalues,
    "volcano_source": volcano_source,
    "position_manifest": position_manifest,
}

for name, df in tables.items():
    print(f"\n{name}")
    print("shape:", df.shape)
    print("columns:", list(df.columns))


wo_counts
shape: (5451, 6)
columns: ['wo', 'umi_count', 'library_id', 'condition', 'replicate', 'sample']

pooled_counts
shape: (2304, 6)
columns: ['library_id', 'condition', 'wo', 'umi_count', 'total_cond', 'freq']

landscape
shape: (768, 11)
columns: ['library_id', 'wo', 'Neg', 'Pos', 'Pre', 'log2fc_pos_pre', 'log2fc_neg_pre', 'delta_pos', 'delta_neg', 'mut_count', 'ddelta']

landscape_pvalues
shape: (768, 23)
columns: ['library_id', 'wo', 'Neg', 'Pos', 'Pre', 'log2fc_pos_pre', 'log2fc_neg_pre', 'delta_pos', 'delta_neg', 'mut_count', 'ddelta', 'count_pre', 'count_pos', 'count_neg', 'p_pos_pre', 'p_neg_pre', 'p_pos_neg', 'q_pos_pre', 'q_neg_pre', 'q_pos_neg', 'p_pos_pre_safe', 'neglog10_p', 'neglog10_p_capped']

volcano_source
shape: (768, 16)
columns: ['library_id', 'wo', 'Neg', 'Pos', 'Pre', 'log2fc_pos_pre', 'log2fc_neg_pre', 'delta_pos', 'delta_neg', 'mut_count', 'ddelta', 'p_pos_neg', 'q_pos_neg', 'p_pos_neg_safe', 'neglog10_p_pos_neg', 'category']

position_manifest
shape: (24,

In [4]:
sample_coverage = (
    wo_counts
    .groupby(["sample", "library_id", "condition", "replicate"], as_index=False)
    .agg(
        total_umi_count=("umi_count", "sum"),
        n_detected_genotypes=("wo", "nunique")
    )
    .sort_values("total_umi_count")
)

sample_coverage

,sample,library_id,condition,replicate,total_umi_count,n_detected_genotypes
11,Library-2-Pos-2,2,Pos,2,440,155
20,Library-3-Pre-2,3,Pre,2,466,191
13,Library-2-Pre-2,2,Pre,2,2240,248
1,Library-1-Neg-2,1,Neg,2,13848,250
17,Library-3-Pos-1,3,Pos,1,14623,255
7,Library-1-Pre-2,1,Pre,2,89573,256
15,Library-3-Neg-2,3,Neg,2,242077,256
4,Library-1-Pos-2,1,Pos,2,338370,256
6,Library-1-Pre-1,1,Pre,1,3048475,256
16,Library-3-Neg-3,3,Neg,3,3426082,256


In [5]:
expected_samples = []
for library_id in [1, 2, 3]:
    for condition in ["Pre", "Pos", "Neg"]:
        for replicate in [1, 2, 3]:
            expected_samples.append({
                "library_id": library_id,
                "condition": condition,
                "replicate": replicate,
                "expected_sample": f"Library-{library_id}-{condition}-{replicate}"
            })

expected_samples = pd.DataFrame(expected_samples)

observed_samples = sample_coverage[["library_id", "condition", "replicate", "sample"]].copy()

sample_check = expected_samples.merge(
    observed_samples,
    on=["library_id", "condition", "replicate"],
    how="left"
)

sample_check["observed_in_wo_counts"] = sample_check["sample"].notna()

known_zero_read_samples = {
    "Library-2-Pos-1",
    "Library-3-Neg-1",
    "Library-3-Pre-1",
}

sample_check["known_zero_read_sample"] = sample_check["expected_sample"].isin(known_zero_read_samples)

sample_check.sort_values(["library_id", "condition", "replicate"])

,library_id,condition,replicate,expected_sample,sample,observed_in_wo_counts,known_zero_read_sample
6,1,Neg,1,Library-1-Neg-1,Library-1-Neg-1,True,False
7,1,Neg,2,Library-1-Neg-2,Library-1-Neg-2,True,False
8,1,Neg,3,Library-1-Neg-3,Library-1-Neg-3,True,False
3,1,Pos,1,Library-1-Pos-1,Library-1-Pos-1,True,False
4,1,Pos,2,Library-1-Pos-2,Library-1-Pos-2,True,False
5,1,Pos,3,Library-1-Pos-3,Library-1-Pos-3,True,False
0,1,Pre,1,Library-1-Pre-1,Library-1-Pre-1,True,False
1,1,Pre,2,Library-1-Pre-2,Library-1-Pre-2,True,False
2,1,Pre,3,Library-1-Pre-3,Library-1-Pre-3,True,False
15,2,Neg,1,Library-2-Neg-1,NaN,False,False


In [6]:
sample_coverage_flagged = sample_coverage.copy()

sample_coverage_flagged["depth_category"] = pd.cut(
    sample_coverage_flagged["total_umi_count"],
    bins=[-1, 0, 10, 1000, 10000, 100000, float("inf")],
    labels=[
        "zero",
        "1-10",
        "11-1,000",
        "1,001-10,000",
        "10,001-100,000",
        ">100,000"
    ]
)

sample_coverage_flagged.sort_values("total_umi_count")

,sample,library_id,condition,replicate,total_umi_count,n_detected_genotypes,depth_category
11,Library-2-Pos-2,2,Pos,2,440,155,"11-1,000"
20,Library-3-Pre-2,3,Pre,2,466,191,"11-1,000"
13,Library-2-Pre-2,2,Pre,2,2240,248,"1,001-10,000"
1,Library-1-Neg-2,1,Neg,2,13848,250,"10,001-100,000"
17,Library-3-Pos-1,3,Pos,1,14623,255,"10,001-100,000"
7,Library-1-Pre-2,1,Pre,2,89573,256,"10,001-100,000"
15,Library-3-Neg-2,3,Neg,2,242077,256,">100,000"
4,Library-1-Pos-2,1,Pos,2,338370,256,">100,000"
6,Library-1-Pre-1,1,Pre,1,3048475,256,">100,000"
16,Library-3-Neg-3,3,Neg,3,3426082,256,">100,000"


In [7]:
coverage_by_library_condition = (
    pooled_counts
    .groupby(["library_id", "condition"], as_index=False)
    .agg(
        total_umi_count=("umi_count", "sum"),
        n_genotypes=("wo", "nunique")
    )
    .sort_values(["library_id", "condition"])
)

coverage_by_library_condition

,library_id,condition,total_umi_count,n_genotypes
0,1,Neg,45002366,256
1,1,Pos,26690098,256
2,1,Pre,31133073,256
3,2,Neg,26845007,256
4,2,Pos,53008735,256
5,2,Pre,15390714,256
6,3,Neg,3668159,256
7,3,Pos,15458642,256
8,3,Pre,29012677,256


In [8]:
landscape_coverage = (
    landscape
    .groupby("library_id", as_index=False)
    .agg(
        n_genotypes=("wo", "nunique"),
        n_rows=("wo", "size")
    )
)

landscape_coverage

,library_id,n_genotypes,n_rows
0,1,256,256
1,2,256,256
2,3,256,256


In [9]:
assert landscape.shape[0] == 3 * 256, "Landscape table should contain 3 libraries x 256 genotypes."
assert landscape["wo"].str.len().eq(8).all(), "All W/O genotypes should have length 8."
assert set(landscape["library_id"]) == {1, 2, 3}, "Expected libraries 1, 2, and 3."

genotypes_per_library = landscape.groupby("library_id")["wo"].nunique()
assert genotypes_per_library.eq(256).all(), "Each library should contain 256 W/O genotypes."

print("Final landscape table passed basic dimensional checks.")

Final landscape table passed basic dimensional checks.
